In [1]:
import warnings
warnings. simplefilter(action='ignore', category=Warning)
import pandas as pd
import numpy as np
from tabulate import tabulate
import yfinance as yf

In [2]:
def GoldenCrossverSignal(stock_symbol):
    data = yf.download(stock_symbol, interval='1d')
    data['Prev_Close'] = data.Close.shift(1)
    data['20_SMA'] = data.Prev_Close.rolling(window=20, min_periods=1).mean()
    data['50_SMA'] = data.Prev_Close.rolling(window=50, min_periods=1).mean()
    data['Signal'] = 0
    data['Signal'] = np.where(data['20_SMA'] > data['50_SMA'], 1, 0)
    data['Position'] = data.Signal.diff()
    df_pos = data[(data['Position'] == 1) | (data['Position'] == -1)].copy()
    df_pos['Position'] = df_pos['Position'].apply(lambda x: 'Buy' if x == 1 else 'Sell')
    return df_pos

In [3]:
data = GoldenCrossverSignal('TATAMOTORS.NS')

[*********************100%%**********************]  1 of 1 completed


In [5]:
data.head()

,Open,High,Low,Close,Adj Close,Volume,Prev_Close,20_SMA,50_SMA,Signal,Position
Date,,,,,,,,,,,
1991-03-08,21.109308,22.157288,20.959597,22.157288,15.905156,0,21.857864,20.345779,20.309785,1,Buy
1991-10-17,38.326118,38.326118,38.326118,38.326118,27.511621,0,38.326118,37.974298,38.095564,0,Sell
1991-11-06,39.972946,39.972946,39.972946,39.972946,28.693764,0,39.972946,38.963890,38.934547,1,Buy
1991-11-27,32.936508,34.134201,32.936508,33.535355,24.072670,0,32.936508,38.153952,38.254857,0,Sell
1992-01-20,34.733047,35.481602,34.733047,35.331890,25.362278,0,35.631313,34.680647,34.511473,1,Buy


In [6]:
required_df = data[(data.index >= data[data['Position'] == 'Buy'].index[0]) & (data.index <= data[data['Position'] == 'Sell'].index[-1])]

In [7]:
required_df.head()

,Open,High,Low,Close,Adj Close,Volume,Prev_Close,20_SMA,50_SMA,Signal,Position
Date,,,,,,,,,,,
1991-03-08,21.109308,22.157288,20.959597,22.157288,15.905156,0,21.857864,20.345779,20.309785,1,Buy
1991-10-17,38.326118,38.326118,38.326118,38.326118,27.511621,0,38.326118,37.974298,38.095564,0,Sell
1991-11-06,39.972946,39.972946,39.972946,39.972946,28.693764,0,39.972946,38.963890,38.934547,1,Buy
1991-11-27,32.936508,34.134201,32.936508,33.535355,24.072670,0,32.936508,38.153952,38.254857,0,Sell
1992-01-20,34.733047,35.481602,34.733047,35.331890,25.362278,0,35.631313,34.680647,34.511473,1,Buy


In [8]:
data[data['Position'] == 'Sell'].index[-1]

Timestamp('2024-05-17 00:00:00')

In [9]:
# Name, Entry TIme, Entry PRice, QTY, Exit Time, Exit Price
class Backtest:
    def __init__(self):
        self.columns = ['Equity Name', 'Trade', 'Entry Time', 'Entry Price', 'Exit Time', 'Exit Price', 'Quantity', 'Position Size', 'PNL', '% PNL']
        self.backtesting = pd.DataFrame(columns=self.columns)

    def buy(self, equity_name, entry_time, entry_price, qty):
        self.trade_log = dict(zip(self.columns, [None] * len(self.columns)))
        self.trade_log['Trade'] = 'Long Open'
        self.trade_log['Quantity'] = qty
        self.trade_log['Position Size'] = round(self.trade_log['Quantity'] * entry_price, 3)
        self.trade_log['Equity Name'] = equity_name
        self.trade_log['Entry Time'] = entry_time
        self.trade_log['Entry Price'] = round(entry_price, 2)

    def sell(self, exit_time, exit_price, exit_type, charge):
        self.trade_log['Trade'] = 'Long Closed'
        self.trade_log['Exit Time'] = exit_time
        self.trade_log['Exit Price'] = round(exit_price, 2)
        self.trade_log['Exit Type'] = exit_type
        self.trade_log['PNL'] = round((self.trade_log['Exit Price'] - self.trade_log['Entry Price']) * self.trade_log['Quantity'] - charge, 3)
        self.trade_log['% PNL'] = round((self.trade_log['PNL'] / self.trade_log['Position Size']) * 100, 3)
        self.trade_log['Holding Period'] = exit_time - self.trade_log['Entry Time']
        self.backtesting = self.backtesting.append(self.trade_log, ignore_index=True)
    
    def stats(self):
        df = self.backtesting
        parameters = ['Total Trade Scripts', 'Total Trade', 'PNL',  'Winners', 'Losers', 'Win Ratio','Total Profit', 'Total Loss', 'Average Loss per Trade', 'Average Profit per Trade', 'Average PNL Per Trade', 'Risk Reward']
        total_traded_scripts = len(df['Equity Name'].unique())
        total_trade = len(df.index)
        pnl = df.PNL.sum()
        winners = len(df[df.PNL > 0])
        loosers = len(df[df.PNL <= 0])
        win_ratio = str(round((winners/total_trade) * 100, 2)) + '%'
        total_profit = round(df[df.PNL > 0].PNL.sum(), 2)
        total_loss  = round(df[df.PNL <= 0].PNL.sum(), 2)
        average_loss_per_trade = round(total_loss/loosers, 2)
        average_profit_per_trade = round(total_profit/winners, 2)
        average_pnl_per_trade = round(pnl/total_trade, 2)
        risk_reward = f'1:{-1 * round(average_profit_per_trade/average_loss_per_trade, 2)}'
        data_points = [total_traded_scripts,total_trade,pnl,winners, loosers, win_ratio, total_profit, total_loss, average_loss_per_trade, average_profit_per_trade, average_pnl_per_trade, risk_reward]
        data = list(zip(parameters,data_points ))
        print(tabulate(data, ['Parameters', 'Values'], tablefmt='psql'))
        

In [ ]:
bt = Backtest()
capital = 50000
scripts =['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'SBIN.NS', 'HDFCBANK.NS', 'HDFCBANK.NS', 'TITAN.NS', 'HEROMOTOCO.NS', 'TATAMOTORS.NS', 'BPCL.NS']

for stock in scripts:
    data = GoldenCrossverSignal(stock)
    required_df = data[(data.index >= data[data['Position'] == 'Buy'].index[0]) & (data.index <= data[data['Position'] == 'Sell'].index[-1])]
    for index, data in required_df.iterrows():
        if(data.Position == 'Buy'):
            qty = capital // data.Open
            bt.buy(stock, index, data.Open, qty)
        else:
            bt.sell(index, data.Open, 'Exit Trigger', 0)

In [13]:
df = bt.backtesting

In [14]:
bt.backtesting.to_csv('Backtest.csv')

In [15]:
bt.stats()

+--------------------------+-------------+
| Parameters               | Values      |
|--------------------------+-------------|
| Total Trade Scripts      | 9           |
| Total Trade              | 758         |
| PNL                      | 2484696.81  |
| Winners                  | 334         |
| Losers                   | 424         |
| Win Ratio                | 44.06%      |
| Total Profit             | 4151421.17  |
| Total Loss               | -1666724.36 |
| Average Loss per Trade   | -3930.95    |
| Average Profit per Trade | 12429.4     |
| Average PNL Per Trade    | 3277.96     |
| Risk Reward              | 1:3.16      |
+--------------------------+-------------+


In [ ]:
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt

class Backtest:
    def __init__(self):
        self.columns = ['Equity Name', 'Trade', 'Entry Time', 'Entry Price', 'Exit Time', 'Exit Price', 'Quantity', 'Position Size', 'PNL', '% PNL', 'Holding Period']
        self.backtesting = pd.DataFrame(columns=self.columns)

    def buy(self, equity_name, entry_time, entry_price, qty, slippage=0, spread=0):
        self.trade_log = dict(zip(self.columns, [None] * len(self.columns)))
        self.trade_log['Trade'] = 'Long Open'
        self.trade_log['Quantity'] = qty
        adjusted_entry_price = entry_price + slippage + (spread / 2)
        self.trade_log['Position Size'] = round(self.trade_log['Quantity'] * adjusted_entry_price, 3)
        self.trade_log['Equity Name'] = equity_name
        self.trade_log['Entry Time'] = entry_time
        self.trade_log['Entry Price'] = round(adjusted_entry_price, 2)

    def sell(self, exit_time, exit_price, exit_type, charge, slippage=0, spread=0):
        adjusted_exit_price = exit_price - slippage - (spread / 2)
        self.trade_log['Trade'] = 'Long Closed'
        self.trade_log['Exit Time'] = exit_time
        self.trade_log['Exit Price'] = round(adjusted_exit_price, 2)
        self.trade_log['Exit Type'] = exit_type
        self.trade_log['PNL'] = round((self.trade_log['Exit Price'] - self.trade_log['Entry Price']) * self.trade_log['Quantity'] - charge, 3)
        self.trade_log['% PNL'] = round((self.trade_log['PNL'] / self.trade_log['Position Size']) * 100, 3)
        self.trade_log['Holding Period'] = exit_time - self.trade_log['Entry Time']
        self.backtesting = self.backtesting.append(self.trade_log, ignore_index=True)

    def short_sell(self, equity_name, entry_time, entry_price, qty):
        self.trade_log = dict(zip(self.columns, [None] * len(self.columns)))
        self.trade_log['Trade'] = 'Short Open'
        self.trade_log['Quantity'] = qty
        self.trade_log['Position Size'] = round(self.trade_log['Quantity'] * entry_price, 3)
        self.trade_log['Equity Name'] = equity_name
        self.trade_log['Entry Time'] = entry_time
        self.trade_log['Entry Price'] = round(entry_price, 2)

    def cover(self, exit_time, exit_price, exit_type, charge):
        self.trade_log['Trade'] = 'Short Closed'
        self.trade_log['Exit Time'] = exit_time
        self.trade_log['Exit Price'] = round(exit_price, 2)
        self.trade_log['Exit Type'] = exit_type
        self.trade_log['PNL'] = round((self.trade_log['Entry Price'] - self.trade_log['Exit Price']) * self.trade_log['Quantity'] - charge, 3)
        self.trade_log['% PNL'] = round((self.trade_log['PNL'] / self.trade_log['Position Size']) * 100, 3)
        self.trade_log['Holding Period'] = exit_time - self.trade_log['Entry Time']
        self.backtesting = self.backtesting.append(self.trade_log, ignore_index=True)

    def stats(self):
        df = self.backtesting
        df['Holding Period'] = df['Holding Period'].apply(lambda x: x.total_seconds() / 3600)  # Convert to hours
        avg_holding_period = df['Holding Period'].mean()

        parameters = ['Total Trade Scripts', 'Total Trade', 'PNL', 'Winners', 'Losers', 'Win Ratio', 'Total Profit', 'Total Loss', 'Average Loss per Trade', 'Average Profit per Trade', 'Average PNL Per Trade', 'Risk Reward', 'Max Drawdown', 'Sharpe Ratio', 'Average Holding Period (hrs)']
        total_traded_scripts = len(df['Equity Name'].unique())
        total_trade = len(df.index)
        pnl = df.PNL.sum()
        winners = len(df[df.PNL > 0])
        losers = len(df[df.PNL <= 0])
        win_ratio = str(round((winners / total_trade) * 100, 2)) + '%'
        total_profit = round(df[df.PNL > 0].PNL.sum(), 2)
        total_loss = round(df[df.PNL <= 0].PNL.sum(), 2)
        average_loss_per_trade = round(total_loss / losers, 2)
        average_profit_per_trade = round(total_profit / winners, 2)
        average_pnl_per_trade = round(pnl / total_trade, 2)
        risk_reward = f'1:{-1 * round(average_profit_per_trade / average_loss_per_trade, 2)}'

        # Calculate max drawdown
        cumulative_pnl = df.PNL.cumsum()
        drawdown = cumulative_pnl - cumulative_pnl.cummax()
        max_drawdown = drawdown.min()

        # Calculate Sharpe Ratio
        mean_pnl = df.PNL.mean()
        pnl_std = df.PNL.std()
        sharpe_ratio = round((mean_pnl / pnl_std) * np.sqrt(252), 2)  # Assuming 252 trading days

        data_points = [total_traded_scripts, total_trade, pnl, winners, losers, win_ratio, total_profit, total_loss, average_loss_per_trade, average_profit_per_trade, average_pnl_per_trade, risk_reward, max_drawdown, sharpe_ratio, avg_holding_period]
        data = list(zip(parameters, data_points))
        print(tabulate(data, ['Parameters', 'Values'], tablefmt='psql'))

    def plot_equity_curve(self):
        df = self.backtesting
        df['Cumulative PNL'] = df.PNL.cumsum()
        plt.figure(figsize=(10, 6))
        plt.plot(df['Exit Time'], df['Cumulative PNL'], label='Equity Curve')
        plt.xlabel('Time')
        plt.ylabel('Cumulative PNL')
        plt.title('Equity Curve')
        plt.legend()
        plt.show()

    def plot_drawdown(self):
        df = self.backtesting
        df['Cumulative PNL'] = df.PNL.cumsum()
        df['Drawdown'] = df['Cumulative PNL'] - df['Cumulative PNL'].cummax()
        plt.figure(figsize=(10, 6))
        plt.plot(df['Exit Time'], df['Drawdown'], label='Drawdown', color='red')
        plt.xlabel('Time')
        plt.ylabel('Drawdown')
        plt.title('Drawdown Plot')
        plt.legend()
        plt.show()


In [ ]:
import pandas as pd
from datetime import datetime

# Create an instance of the Backtest class
backtest = Backtest()

# Example 1: Long Trade
backtest.buy(equity_name="AAPL", entry_time=datetime(2023, 1, 10, 10, 0), entry_price=150, qty=10)
backtest.sell(exit_time=datetime(2023, 1, 15, 15, 0), exit_price=155, exit_type='Market', charge=5)

# Example 2: Long Trade with Slippage and Spread
backtest.buy(equity_name="MSFT", entry_time=datetime(2023, 2, 5, 11, 0), entry_price=250, qty=20, slippage=0.5, spread=0.1)
backtest.sell(exit_time=datetime(2023, 2, 10, 13, 0), exit_price=260, exit_type='Market', charge=10, slippage=0.5, spread=0.1)

# Example 3: Short Trade
backtest.short_sell(equity_name="TSLA", entry_time=datetime(2023, 3, 1, 9, 0), entry_price=700, qty=5)
backtest.cover(exit_time=datetime(2023, 3, 10, 16, 0), exit_price=680, exit_type='Market', charge=7)

# Example 4: Short Trade with Slippage and Spread
backtest.short_sell(equity_name="GOOGL", entry_time=datetime(2023, 4, 3, 14, 0), entry_price=1500, qty=3)
backtest.cover(exit_time=datetime(2023, 4, 20, 10, 0), exit_price=1450, exit_type='Market', charge=15)

# Display Statistics
backtest.stats()

# Plot Equity Curve
backtest.plot_equity_curve()

# Plot Drawdown
backtest.plot_drawdown()
